In [23]:
import sys
my_framework_path = '/home/joao/Área de trabalho/IC_2025.2/repositorios/metapy/'
#my_framework_path = r'C:\git-projetos\metapy' # PC prof. wanderlei
sys.path.append(my_framework_path)
from metapy_toolbox import *


In [65]:
import numpy as np
import pandas as pd
from typing import Callable, Optional, Union, Tuple

def update_velocity_pso(w: float, c1: float, c2: float,
                    current_velocity: list, current_position: list,
                    p_best_position: list, g_best_position: list) -> Tuple[list, str]:
    report_move = "    Atualização de Velocidade\n"
    report_move += f"    v_atual = {current_velocity}\n"
    report_move += f"    x_atual = {current_position}\n"
    new_velocity = []
    d = len(current_position)

    r1 = np.random.uniform(low=0, high=1, size=d)
    r2 = np.random.uniform(low=0, high=1, size=d)

    for j in range(d):
        inertia = w * current_velocity[j]
        cognitive = c1 * r1[j] * (p_best_position[j] - current_position[j])
        social = c2 * r2[j] * (g_best_position[j] - current_position[j])
        v_j = inertia + cognitive + social
        new_velocity.append(v_j)
        report_move += f"    Dimensão {j}: v_novo = {inertia:.3f} (inercia) + {cognitive:.3f} (cognitivo) + {social:.3f} (social) = {v_j:.3f}\n"

    report_move += f"    Velocidade final nova = {new_velocity}\n"
    return new_velocity, report_move


def update_position_pso(current_position: list, new_velocity: list,
                    x_lower: list, x_upper: list) -> Tuple[list, str]: # Corrigido aqui
    report_move = "    Atualização de Posição\n"
    report_move += f"    x_atual = {current_position}\n"
    report_move += f"    v_aplicada = {new_velocity}\n"

    x_new = (np.array(current_position) + np.array(new_velocity)).tolist()
    report_move += f"    x antes da verificação de limites = {x_new}\n"

    x_new_checked = funcs.check_interval_01(x_new, x_lower, x_upper)

    report_move += f"    Posição final nova = {x_new_checked}\n"
    return x_new_checked, report_move


def particle_swarm_optimization_01(obj: Callable, n_gen: int, params: dict, initial_population: list, x_lower: list, x_upper: list, args: Optional[Tuple] = None, robustness: Union[bool, dict] = False) -> Tuple[pd.DataFrame, pd.DataFrame, str]:
    x_t0 = initial_population.copy()
    d = len(x_t0[0])
    n_pop = len(x_t0)
    all_results = []
    bests = []
    report = "Particle Swarm Optimization\n"
    w_start = params['inertia weight']['w_start']
    w_end = params['inertia weight']['w_end']
    c1 = params['cognitive']['c1']
    c2 = params['social']['c2']
    velocity_init_type = params['velocity']['type'].lower()

    v_t = []
    if velocity_init_type == 'random':
        v_max = params['velocity']['vmax']
        v_min = params['velocity']['vmin']
        for _ in range(n_pop):
            v_t.append(np.random.uniform(low=v_min, high=v_max, size=d).tolist())
    else:
        for _ in range(n_pop):
            v_t.append([0.0] * d)
            
    # Laço da população inicial
    for n in range(n_pop):
        # --- LINHA CORRIGIDA AQUI --- 👇
        aux_df = funcs.evaluation(obj, n, x_t0[n], n, 0, args=args) if args is not None else funcs.evaluation(obj, n, x_t0[n], n, 0)
        all_results.append(aux_df)
        
    df = pd.concat(all_results, ignore_index=True)
    df['REPORT'] = ""
    df['W'] = np.nan
    df.loc[df['ITER'] == 0, 'W'] = w_start
    
    for j in range(d):
        df.loc[:, f'P_X_BEST_{j}'] = df.loc[:, f'X_{j}']
    df.loc[:, 'P_OF_BEST'] = df.loc[:, 'OF']
    
    g_best_row = df.loc[df['P_OF_BEST'].idxmin()]
    g_best_x = [g_best_row[f'P_X_BEST_{j}'] for j in range(d)]
    g_best_of = g_best_row['P_OF_BEST']
    report += f"Initial Global Best OF: {g_best_of}\n"

    for t in range(1, n_gen + 1):
        w = w_start - (w_start - w_end) * (t / n_gen)
        report += f"iteration: {t}, w = {w:.4f}\n"
        df_aux = df[df['ITER'] == t-1].reset_index(drop=True)
        aux_t_dfs = []
        bests.append(funcs.best_avg_worst(df_aux, d))
        
        g_best_row_candidate = df_aux.loc[df_aux['P_OF_BEST'].idxmin()]
        if g_best_row_candidate['P_OF_BEST'] < g_best_of:
            g_best_of = g_best_row_candidate['P_OF_BEST']
            g_best_x = [g_best_row_candidate[f'P_X_BEST_{j}'] for j in range(d)]
        
        for i in range(n_pop):
            x_current, _, _ = funcs.query_x_of_fit_from_data(df_aux, i, d)
            p_best_x = [df_aux.loc[i, f'P_X_BEST_{j}'] for j in range(d)]
            current_velocity = v_t[i]
            
            new_velocity, _ = update_velocity_pso(w, c1, c2, current_velocity, x_current, p_best_x, g_best_x)
            x_new, _ = update_position_pso(x_current, new_velocity, x_lower, x_upper)
            v_t[i] = new_velocity
            
            n_evals_total = len(df)
            aux_df_eval = funcs.evaluation(obj, i, x_new, n_evals_total, t, args=args)
            aux_t_dfs.append(aux_df_eval)

        df_current_iter = pd.concat(aux_t_dfs, ignore_index=True)
        df_current_iter['W'] = w
        
        df_before_concat = df.copy()
        
        df = pd.concat([df_before_concat, df_current_iter], ignore_index=True)

        df_past = df_before_concat[df_before_concat['ITER'] == t-1].reset_index(drop=True)
        df_current = df[df['ITER'] == t].reset_index(drop=True)
        
        new_p_bests = []
        for i in range(n_pop):
            past_of = df_past.loc[i, 'P_OF_BEST']
            current_of = df_current.loc[i, 'OF']
            
            p_best_row = {}
            if current_of < past_of:
                p_best_row['P_OF_BEST'] = current_of
                for j in range(d):
                    p_best_row[f'P_X_BEST_{j}'] = df_current.loc[i, f'X_{j}']
            else:
                p_best_row['P_OF_BEST'] = past_of
                for j in range(d):
                    p_best_row[f'P_X_BEST_{j}'] = df_past.loc[i, f'P_X_BEST_{j}']
            new_p_bests.append(p_best_row)

        df_new_p_bests = pd.DataFrame(new_p_bests)
        df.loc[df['ITER'] == t, df_new_p_bests.columns] = df_new_p_bests.values


    dfj = df[df['ITER'] == n_gen].reset_index(drop=True)
    bests.append(funcs.best_avg_worst(dfj, d))
    df_resume = pd.concat(bests, ignore_index=True)
    df['REPORT'] = report
    
    return df, df_resume, report


In [ ]:
def active_learning_example(x, args=None):
    """
    Função de teste: Parábola deslocada.
    f(x) = (x - 15)^2 + 7
    """
    valor_a_minimizar = (x[0] - 15)**2 + 7
    
    return valor_a_minimizar



In [67]:
params_pso = {
    'inertia weight': {'w_start': 0.9, 'w_end': 0.4},
    'cognitive': {'c1': 2.05},
    'social': {'c2': 2.05},
    'velocity': {'type': 'random', 'vmin': -2, 'vmax': 2}
}

x_ini = [
    [0.0],
    [7.0],
    [25.0]
]
n_gen = 50

df, df_resume, report = particle_swarm_optimization_01(
    obj=active_learning_example,
    n_gen=n_gen,
    params=params_pso,
    initial_population=x_ini,
    x_lower=[0.0],
    x_upper=[25.0]
)

df


,ID,ITER,X_0,OF,FIT,OF EVALUATIONS,TIME CONSUMPTION (s),REPORT,W,P_X_BEST_0,P_OF_BEST
0,0,0,0.000000,232.000000,0.004292,1,0.000003,Particle Swarm Optimization\nInitial Global Be...,0.90,0.000000,232.000000
1,1,0,7.000000,71.000000,0.013889,1,0.000002,Particle Swarm Optimization\nInitial Global Be...,0.90,7.000000,71.000000
2,2,0,25.000000,107.000000,0.009259,1,0.000001,Particle Swarm Optimization\nInitial Global Be...,0.90,25.000000,107.000000
3,0,1,3.081594,149.048402,0.006665,1,0.000003,Particle Swarm Optimization\nInitial Global Be...,0.89,3.081594,149.048402
4,1,1,7.686155,60.492326,0.016262,1,0.000003,Particle Swarm Optimization\nInitial Global Be...,0.89,7.686155,60.492326
...,...,...,...,...,...,...,...,...,...,...,...
148,1,49,15.033377,7.001114,0.124983,1,0.000001,Particle Swarm Optimization\nInitial Global Be...,0.41,15.000751,7.000001
149,2,49,15.041898,7.001755,0.124973,1,0.000001,Particle Swarm Optimization\nInitial Global Be...,0.41,14.999610,7.000000
150,0,50,15.006584,7.000043,0.124999,1,0.000003,Particle Swarm Optimization\nInitial Global Be...,0.40,14.996192,7.000014
151,1,50,14.993542,7.000042,0.124999,1,0.000002,Particle Swarm Optimization\nInitial Global Be...,0.40,15.000751,7.000001


In [60]:
report


'Particle Swarm Optimization\nInitial Global Best OF: 71.0\niteration: 1, w = 0.8833\niteration: 2, w = 0.8667\niteration: 3, w = 0.8500\niteration: 4, w = 0.8333\niteration: 5, w = 0.8167\niteration: 6, w = 0.8000\niteration: 7, w = 0.7833\niteration: 8, w = 0.7667\niteration: 9, w = 0.7500\niteration: 10, w = 0.7333\niteration: 11, w = 0.7167\niteration: 12, w = 0.7000\niteration: 13, w = 0.6833\niteration: 14, w = 0.6667\niteration: 15, w = 0.6500\niteration: 16, w = 0.6333\niteration: 17, w = 0.6167\niteration: 18, w = 0.6000\niteration: 19, w = 0.5833\niteration: 20, w = 0.5667\niteration: 21, w = 0.5500\niteration: 22, w = 0.5333\niteration: 23, w = 0.5167\niteration: 24, w = 0.5000\niteration: 25, w = 0.4833\niteration: 26, w = 0.4667\niteration: 27, w = 0.4500\niteration: 28, w = 0.4333\niteration: 29, w = 0.4167\niteration: 30, w = 0.4000\n'